<details>
<summary><b>Step 1: Import packages and load the node layout</b></summary>

This step imports the required Python packages and loads the `Nodes.xlsx` file.  
The `Node` sheet contains each location/node, its coordinates, type, and whether it requires centerline logic.

The code creates a `nodes` dictionary where each node has:
- a node ID
- x/y coordinates
- location type
- centerline requirement
</details>

In [22]:
import pandas as pd
import numpy as np
import heapq

# Set the main project folder path
rootpath = r"C:\Users\Kaitlin Gilleo\OneDrive\Documents\GitHub\BSC-FGI_Scheduler"

# Load the Node sheet from Nodes.xlsx
nodes_xl = pd.read_excel(
    rootpath + r"\input\Nodes.xlsx",
    sheet_name="Node",
    engine="openpyxl"
)

# Remove extra spaces from column names
nodes_xl.columns = nodes_xl.columns.str.strip()

# Keep only rows that have a valid node ID and coordinates
valid_node_rows = nodes_xl[
    nodes_xl["node_id"].notna() &
    nodes_xl["x"].notna() &
    nodes_xl["y"].notna()
].copy()

# Build a dictionary of nodes and their attributes
nodes = {}

for _, row in valid_node_rows.iterrows():
    node_id = str(row["node_id"]).strip()

    nodes[node_id] = {
        "id": node_id,
        "coord": [float(row["x"]), float(row["y"])],
        "type": row["type"] if "type" in nodes_xl.columns else None,
        "req_centerline": row["req_centerline"] if "req_centerline" in nodes_xl.columns else None
    }

print(f"Nodes loaded: {len(nodes)}")

Nodes loaded: 80


<details>
<summary><b>Step 2: Build the adjacency / traffic-flow map</b></summary>

This step loads the `adjacency` sheet, which defines which locations are connected.

Most connections are treated as two-way. However, the FGI row needs special one-way logic:

P3SW → FGI1 → FGI2 → FGI3 → FGI4 → N38

This prevents the routing model from taking shortcuts into FGI4 through N38/N37/P3SW.
</details>

In [23]:
# Load the adjacency sheet from Nodes.xlsx
adjacency_xl = pd.read_excel(
    rootpath + r"\input\Nodes.xlsx",
    sheet_name="adjacency",
    engine="openpyxl"
)

# Remove extra spaces from column names
adjacency_xl.columns = adjacency_xl.columns.str.strip()

# Create an empty neighbor map for every node
# This will store which nodes each location can move to
neighbor_map = {node_id: [] for node_id in nodes}

# These are the one-way directions that represent the required FGI traffic flow
allowed_one_way_edges = {
    ("P3SW", "FGI1"),
    ("FGI1", "FGI2"),
    ("FGI2", "FGI3"),
    ("FGI3", "FGI4"),
    ("FGI4", "N38")
}

# These directions should never be allowed
# This blocks reverse movement through the FGI row
blocked_edges = {
    ("FGI1", "P3SW"),
    ("FGI2", "FGI1"),
    ("FGI3", "FGI2"),
    ("FGI4", "FGI3"),
    ("N38", "FGI4")
}

# Keep only valid adjacency rows
valid_edge_rows = adjacency_xl[
    adjacency_xl["from_node"].notna() &
    adjacency_xl["to_node"].notna()
].copy()

# Build the traffic-flow map
for _, row in valid_edge_rows.iterrows():
    u = str(row["from_node"]).strip()
    v = str(row["to_node"]).strip()

    # Only use edges where both nodes exist
    if u in nodes and v in nodes and u != v:

        # Add movement from u to v unless that direction is blocked
        if (u, v) not in blocked_edges:
            neighbor_map[u].append(v)

        # Add movement from v to u unless the reverse direction is blocked
        if (v, u) not in blocked_edges:
            neighbor_map[v].append(u)

# Remove duplicate neighbors
for node_id in neighbor_map:
    neighbor_map[node_id] = list(dict.fromkeys(neighbor_map[node_id]))

print("Neighbor map created.")
print(f"Edges loaded: {len(valid_edge_rows)}")

# Print key FGI connections to confirm one-way logic worked
for node in ["P3SW", "FGI1", "FGI2", "FGI3", "FGI4", "N38", "N37"]:
    print(node, "->", neighbor_map.get(node, []))

Neighbor map created.
Edges loaded: 84
P3SW -> ['N37']
FGI1 -> ['N4', 'FGI2']
FGI2 -> ['FGI3']
FGI3 -> ['FGI4']
FGI4 -> ['N38']
N38 -> ['N37', 'N39', 'N40']
N37 -> ['P3SW', 'N38']


<details>
<summary><b>Step 3: Define routing and move-time functions</b></summary>

This step defines the functions used to calculate routes and move times.

The routing function finds the shortest valid path from one location to another using the allowed traffic-flow connections.

Move time is calculated using:

distance in feet → distance in miles → travel time in hours

The default speed is set to 3 mph.
</details>

In [24]:
def point_distance(nodes, a, b):
    """
    Calculate straight-line distance between two connected nodes.
    Coordinates are assumed to be in feet.
    """
    x1, y1 = nodes[a]["coord"]
    x2, y2 = nodes[b]["coord"]

    return ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5


def greedy_route(nodes, neighbor_map, origin, destination):
    """
    Find the shortest valid route from origin to destination.

    This uses Dijkstra's algorithm:
    - checks all allowed connected paths
    - respects one-way blocked FGI movement
    - returns the shortest total distance
    """
    # If either node does not exist, no route is possible
    if origin not in nodes or destination not in nodes:
        return None

    # If origin and destination are the same, distance is zero
    if origin == destination:
        return {"total_distance": 0.0}

    # Store the best known distance to each node
    best_distance = {origin: 0.0}

    # Priority queue for checking shortest paths first
    frontier = [(0.0, origin)]

    while frontier:
        current_distance, current = heapq.heappop(frontier)

        # Stop once the destination is reached
        if current == destination:
            break

        # Check all locations that can be reached from the current node
        for next_node in neighbor_map.get(current, []):
            segment_distance = point_distance(nodes, current, next_node)
            new_distance = current_distance + segment_distance

            # Update shortest known distance if this route is better
            if new_distance < best_distance.get(next_node, np.inf):
                best_distance[next_node] = new_distance
                heapq.heappush(frontier, (new_distance, next_node))

    # If destination was never reached, return no route
    if destination not in best_distance:
        return None

    return {"total_distance": best_distance[destination]}


def calc_move_time(route_info, speed_mph=3):
    """
    Convert route distance into move time.

    Output is in hours.
    """
    if route_info is None:
        return np.inf

    distance_feet = route_info["total_distance"]
    distance_miles = distance_feet / 5280

    return distance_miles / speed_mph

<details>
<summary><b>Step 4: Build the distance and move-time matrices</b></summary>

This step calculates the shortest valid route between every pair of locations.

It creates two matrices:
- `distance_matrix`: total distance between each location pair
- `move_time_matrix`: move time between each location pair

These matrices use the corrected FGI traffic flow, so moves into FGI must enter through FGI1.
</details>

In [25]:
# Get all node IDs
node_ids = list(nodes.keys())

# Create blank matrices filled with infinity
# Infinity means no valid path has been found yet
distance_matrix = pd.DataFrame(np.inf, index=node_ids, columns=node_ids)
move_time_matrix = pd.DataFrame(np.inf, index=node_ids, columns=node_ids)

# Calculate distance and move time for every origin-destination pair
for origin in node_ids:
    for destination in node_ids:

        # Find the shortest valid route
        route_info = greedy_route(nodes, neighbor_map, origin, destination)

        # Same-location moves have zero distance and zero time
        if origin == destination:
            distance_matrix.loc[origin, destination] = 0.0
            move_time_matrix.loc[origin, destination] = 0.0

        # If a valid route exists, calculate distance and move time
        elif route_info is not None:
            distance_matrix.loc[origin, destination] = route_info["total_distance"]
            move_time_matrix.loc[origin, destination] = calc_move_time(route_info, speed_mph=3)

print("Move-time matrix built.")

# Display the move-time matrix
move_time_matrix

Move-time matrix built.


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,N36,P3SW,FGI1,FGI2,FGI3,FGI4,N37,N38,N39,N40
A1,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,0.113636,0.129419,0.145202,0.160985,...,0.217172,0.193813,0.113321,0.130682,0.148043,0.165404,0.186869,0.172348,0.145833,0.195076
A2,0.034722,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,0.113636,0.129419,0.145202,...,0.201389,0.178030,0.097538,0.114899,0.132260,0.149621,0.171086,0.156566,0.130051,0.179293
A3,0.050505,0.034722,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,0.113636,0.129419,...,0.185606,0.162247,0.081755,0.099116,0.116477,0.133838,0.155303,0.140783,0.114268,0.163510
A4,0.066288,0.050505,0.034722,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,0.113636,...,0.169823,0.146465,0.072285,0.089646,0.107008,0.124369,0.139520,0.125000,0.098485,0.147727
A5,0.082071,0.066288,0.050505,0.034722,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,...,0.154040,0.130682,0.088068,0.105429,0.122790,0.140152,0.123737,0.109217,0.082702,0.131944
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FGI4,0.179293,0.163510,0.147727,0.131944,0.116162,0.100379,0.084596,0.068813,0.053030,0.048611,...,0.051768,0.028409,0.185290,0.202652,0.220013,0.000000,0.021465,0.006944,0.033460,0.029672
N37,0.186869,0.171086,0.155303,0.139520,0.123737,0.107955,0.092172,0.076389,0.060606,0.056187,...,0.059343,0.006944,0.192866,0.210227,0.227588,0.244949,0.000000,0.014520,0.041035,0.037247
N38,0.172348,0.156566,0.140783,0.125000,0.109217,0.093434,0.077652,0.061869,0.046086,0.041667,...,0.044823,0.021465,0.178346,0.195707,0.213068,0.230429,0.014520,0.000000,0.026515,0.022727
N39,0.145833,0.130051,0.114268,0.098485,0.082702,0.066919,0.051136,0.035354,0.019571,0.015152,...,0.071338,0.047980,0.151831,0.169192,0.186553,0.203914,0.041035,0.026515,0.000000,0.049242


<details>
<summary><b>Step 5: Export the move-time matrix</b></summary>

This step exports the final matrices to Excel.

The output file contains:
- `location_move_times`: move time between each pair of locations
- `distance_matrix`: distance between each pair of locations

This file can be used as an input for the scheduler or optimization model.
</details>

In [26]:
# Set output file path
output_file = rootpath + r"\output\location_move_times.xlsx"

# Copy matrices for reporting
report_move_times = move_time_matrix.copy()
report_distance = distance_matrix.copy()

# Name the index column for Excel export
report_move_times.index.name = "from_loc"
report_distance.index.name = "from_loc"

# Export both matrices to Excel
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    report_move_times.reset_index().to_excel(
        writer,
        sheet_name="location_move_times",
        index=False
    )

    report_distance.reset_index().to_excel(
        writer,
        sheet_name="distance_matrix",
        index=False
    )

print(f"Saved to: {output_file}")

# Display final move-time matrix
report_move_times

Saved to: C:\Users\Kaitlin Gilleo\OneDrive\Documents\GitHub\BSC-FGI_Scheduler\output\location_move_times.xlsx


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,N36,P3SW,FGI1,FGI2,FGI3,FGI4,N37,N38,N39,N40
from_loc,,,,,,,,,,,,,,,,,,,,,
A1,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,0.113636,0.129419,0.145202,0.160985,...,0.217172,0.193813,0.113321,0.130682,0.148043,0.165404,0.186869,0.172348,0.145833,0.195076
A2,0.034722,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,0.113636,0.129419,0.145202,...,0.201389,0.178030,0.097538,0.114899,0.132260,0.149621,0.171086,0.156566,0.130051,0.179293
A3,0.050505,0.034722,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,0.113636,0.129419,...,0.185606,0.162247,0.081755,0.099116,0.116477,0.133838,0.155303,0.140783,0.114268,0.163510
A4,0.066288,0.050505,0.034722,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,0.113636,...,0.169823,0.146465,0.072285,0.089646,0.107008,0.124369,0.139520,0.125000,0.098485,0.147727
A5,0.082071,0.066288,0.050505,0.034722,0.000000,0.034722,0.050505,0.066288,0.082071,0.097854,...,0.154040,0.130682,0.088068,0.105429,0.122790,0.140152,0.123737,0.109217,0.082702,0.131944
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FGI4,0.179293,0.163510,0.147727,0.131944,0.116162,0.100379,0.084596,0.068813,0.053030,0.048611,...,0.051768,0.028409,0.185290,0.202652,0.220013,0.000000,0.021465,0.006944,0.033460,0.029672
N37,0.186869,0.171086,0.155303,0.139520,0.123737,0.107955,0.092172,0.076389,0.060606,0.056187,...,0.059343,0.006944,0.192866,0.210227,0.227588,0.244949,0.000000,0.014520,0.041035,0.037247
N38,0.172348,0.156566,0.140783,0.125000,0.109217,0.093434,0.077652,0.061869,0.046086,0.041667,...,0.044823,0.021465,0.178346,0.195707,0.213068,0.230429,0.014520,0.000000,0.026515,0.022727


<details>
<summary><b>Step 6: Compare modeled move times to historical move times</b></summary>

This step loads historical move-time records and compares them to the modeled move times.

The process:
- loads only the `Move Times` sheet
- keeps only start, move time, and end columns
- removes centerline moves
- removes outliers from historical move times
- merges historical and modeled move times
- calculates model error
- computes an overall correction factor to calibrate the move-time matrix
</details>

In [27]:
# Load historical move-time file
hist_file = rootpath + r"\input\Centerlines and Move Times Purdue.xlsx"

hist_xl = pd.read_excel(
    hist_file,
    sheet_name="Move Times",
    engine="openpyxl"
)

# Clean column names
hist_xl.columns = hist_xl.columns.str.strip()

# Keep relevant columns and rename them
hist = hist_xl[["Starting Position", "Move Time", "Ending Position", "Centerline"]].copy()

hist = hist.rename(columns={
    "Starting Position": "start",
    "Move Time": "historical_move_time",
    "Ending Position": "end"
})

# Remove centerline moves
hist = hist[hist["Centerline"].fillna("").astype(str).str.upper() != "Y"].copy()

# Keep only needed columns
hist = hist[["start", "historical_move_time", "end"]]

# Clean text values
hist["start"] = hist["start"].astype(str).str.strip()
hist["end"] = hist["end"].astype(str).str.strip()

# Make sure move time is numeric
hist["historical_move_time"] = pd.to_numeric(
    hist["historical_move_time"],
    errors="coerce"
)

# Remove missing move times
hist = hist.dropna(subset=["historical_move_time"])

# Remove outliers using IQR
Q1 = hist["historical_move_time"].quantile(0.25)
Q3 = hist["historical_move_time"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

hist_clean = hist[
    (hist["historical_move_time"] >= lower_bound) &
    (hist["historical_move_time"] <= upper_bound)
].copy()

print(f"Original historical rows: {len(hist)}")
print(f"Rows after outlier removal: {len(hist_clean)}")
print(f"Outliers removed: {len(hist) - len(hist_clean)}")
print(f"Outlier bounds: {lower_bound:.4f} to {upper_bound:.4f} hours")

hist_clean.head()

Original historical rows: 90
Rows after outlier removal: 73
Outliers removed: 17
Outlier bounds: 0.1250 to 0.4583 hours


,start,historical_move_time,end
3,A9,0.166667,BSC1
9,BSC1,0.166667,A9
12,P3,0.333333,A1
13,C2,0.250000,L4
14,D2,0.250000,C2


In [28]:
# Convert modeled move-time matrix into long format
model_long = (
    move_time_matrix
    .reset_index()
    .rename(columns={"index": "start"})
    .melt(id_vars="start", var_name="end", value_name="modeled_move_time")
)

# Remove same-location rows
model_long = model_long[model_long["start"] != model_long["end"]].copy()

# Merge historical data with modeled move times
comparison = hist_clean.merge(
    model_long,
    on=["start", "end"],
    how="inner"
)

# Remove invalid modeled move times
comparison = comparison[np.isfinite(comparison["modeled_move_time"])].copy()

# Calculate original model error
comparison["error"] = comparison["modeled_move_time"] - comparison["historical_move_time"]
comparison["abs_error"] = comparison["error"].abs()
comparison["pct_error"] = comparison["error"] / comparison["historical_move_time"]

mae_original = comparison["abs_error"].mean()
rmse_original = np.sqrt((comparison["error"] ** 2).mean())
mape_original = comparison["pct_error"].abs().mean()

print(f"Matched historical/model rows: {len(comparison)}")
print("\nOriginal Model Error Metrics")
print(f"MAE:  {mae_original:.4f} hours")
print(f"RMSE: {rmse_original:.4f} hours")
print(f"MAPE: {mape_original:.2%}")

comparison.head()

Matched historical/model rows: 58

Original Model Error Metrics
MAE:  0.1276 hours
RMSE: 0.1478 hours
MAPE: 45.88%


,start,historical_move_time,end,modeled_move_time,error,abs_error,pct_error
0,A9,0.166667,BSC1,0.094066,-0.072601,0.072601,-0.435606
1,BSC1,0.166667,A9,0.094066,-0.072601,0.072601,-0.435606
2,C2,0.250000,L4,0.264378,0.014378,0.014378,0.057512
3,D2,0.250000,C2,0.186869,-0.063131,0.063131,-0.252525
4,D1,0.166667,L4,0.123781,-0.042886,0.042886,-0.257316


In [29]:
from sklearn.linear_model import LinearRegression

# Fit linear calibration:
# historical move time = intercept + slope * modeled move time
X = comparison[["modeled_move_time"]]
y = comparison["historical_move_time"]

linear_model = LinearRegression()
linear_model.fit(X, y)

intercept = linear_model.intercept_
slope = linear_model.coef_[0]

print("Linear Calibration Equation")
print(f"historical_move_time = {intercept:.4f} + {slope:.4f} * modeled_move_time")

# Apply linear calibration to the full move-time matrix
linear_calibrated_move_time_matrix = (move_time_matrix * slope) + intercept

# Keep same-location moves equal to 0
for loc in linear_calibrated_move_time_matrix.index:
    linear_calibrated_move_time_matrix.loc[loc, loc] = 0

# Convert calibrated matrix into long format
linear_long = (
    linear_calibrated_move_time_matrix
    .reset_index()
    .rename(columns={"index": "start"})
    .melt(id_vars="start", var_name="end", value_name="linear_calibrated_move_time")
)

linear_long = linear_long[linear_long["start"] != linear_long["end"]].copy()

# Merge historical data with calibrated modeled move times
comparison_linear = hist_clean.merge(
    linear_long,
    on=["start", "end"],
    how="inner"
)

comparison_linear = comparison_linear[
    np.isfinite(comparison_linear["linear_calibrated_move_time"])
].copy()

# Calculate calibrated model error
comparison_linear["error"] = (
    comparison_linear["linear_calibrated_move_time"] -
    comparison_linear["historical_move_time"]
)

comparison_linear["abs_error"] = comparison_linear["error"].abs()
comparison_linear["pct_error"] = (
    comparison_linear["error"] /
    comparison_linear["historical_move_time"]
)

mae_linear = comparison_linear["abs_error"].mean()
rmse_linear = np.sqrt((comparison_linear["error"] ** 2).mean())
mape_linear = comparison_linear["pct_error"].abs().mean()

print("\nLinear Calibration Error Metrics")
print(f"MAE:  {mae_linear:.4f} hours")
print(f"RMSE: {rmse_linear:.4f} hours")
print(f"MAPE: {mape_linear:.2%}")

comparison_linear.head()

Linear Calibration Equation
historical_move_time = 0.2006 + 0.5495 * modeled_move_time

Linear Calibration Error Metrics
MAE:  0.0507 hours
RMSE: 0.0630 hours
MAPE: 19.74%


,start,historical_move_time,end,linear_calibrated_move_time,error,abs_error,pct_error
0,A9,0.166667,BSC1,0.252297,0.085630,0.085630,0.513781
1,BSC1,0.166667,A9,0.252297,0.085630,0.085630,0.513781
2,C2,0.250000,L4,0.345887,0.095887,0.095887,0.383549
3,D2,0.250000,C2,0.303294,0.053294,0.053294,0.213177
4,D1,0.166667,L4,0.268626,0.101959,0.101959,0.611755


In [31]:
# Export calibrated move-time matrix
calibrated_output = rootpath + r"\output\move_time_estimation.xlsx"

report_linear_calibrated = linear_calibrated_move_time_matrix.copy()
report_linear_calibrated.index.name = "from_loc"

with pd.ExcelWriter(calibrated_output, engine="openpyxl") as writer:
    report_linear_calibrated.reset_index().to_excel(
        writer,
        sheet_name="location_move_times",
        index=False
    )

print(f"Saved calibrated matrix to: {calibrated_output}")

report_linear_calibrated

Saved calibrated matrix to: C:\Users\Kaitlin Gilleo\OneDrive\Documents\GitHub\BSC-FGI_Scheduler\output\move_time_estimation.xlsx


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,...,N36,P3SW,FGI1,FGI2,FGI3,FGI4,N37,N38,N39,N40
from_loc,,,,,,,,,,,,,,,,,,,,,
A1,0.000000,0.219686,0.228359,0.237032,0.245705,0.254378,0.263051,0.271724,0.280397,0.289070,...,0.319946,0.307110,0.262878,0.272418,0.281959,0.291499,0.303294,0.295315,0.280744,0.307804
A2,0.219686,0.000000,0.219686,0.228359,0.237032,0.245705,0.254378,0.263051,0.271724,0.280397,...,0.311273,0.298437,0.254205,0.263745,0.273285,0.282826,0.294621,0.286642,0.272071,0.299131
A3,0.228359,0.219686,0.000000,0.219686,0.228359,0.237032,0.245705,0.254378,0.263051,0.271724,...,0.302600,0.289764,0.245532,0.255072,0.264612,0.274153,0.285948,0.277969,0.263398,0.290458
A4,0.237032,0.228359,0.219686,0.000000,0.219686,0.228359,0.237032,0.245705,0.254378,0.263051,...,0.293927,0.281091,0.240328,0.249868,0.259409,0.268949,0.277275,0.269296,0.254725,0.281785
A5,0.245705,0.237032,0.228359,0.219686,0.000000,0.219686,0.228359,0.237032,0.245705,0.254378,...,0.285254,0.272418,0.249001,0.258541,0.268082,0.277622,0.268602,0.260623,0.246052,0.273112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FGI4,0.299131,0.290458,0.281785,0.273112,0.264439,0.255766,0.247093,0.238420,0.229747,0.227318,...,0.229053,0.216217,0.302427,0.311967,0.321508,0.000000,0.212401,0.204422,0.218992,0.216911
N37,0.303294,0.294621,0.285948,0.277275,0.268602,0.259929,0.251256,0.242583,0.233910,0.231482,...,0.233216,0.204422,0.306590,0.316130,0.325671,0.335211,0.000000,0.208585,0.223155,0.221074
N38,0.295315,0.286642,0.277969,0.269296,0.260623,0.251950,0.243277,0.234604,0.225931,0.223502,...,0.225237,0.212401,0.298611,0.308151,0.317691,0.327232,0.208585,0.000000,0.215176,0.213095
